# OCR Quality Check

#### 1. Imports

In [8]:
import pandas as pd
from lxml import etree
import re
from jiwer import cer  # pip install jiwer
import os

#### 2. I am using the same Parsing function as in 02_xml_parsing.ipynb

In [9]:
namespace = {"pc": "http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15"}

def parse_one_file(filepath, namespace):
    rows = []
    tree = etree.parse(filepath)
    root = tree.getroot()
    regions = root.findall(".//pc:TextRegion", namespace)
    for region in regions:
        custom = region.get("custom", "")
        type_match = re.search(r"type:([^;]+);", custom)
        region_type = type_match.group(1) if type_match else "unknown"
        unicode_element = region.find(".//pc:Unicode", namespace)
        if unicode_element is not None and unicode_element.text is not None:
            text = unicode_element.text.strip()
            rows.append({
                "source_file": filepath,
                "region_type": region_type,
                "text":        text
            })
    return rows

#### 3. Parsing the Ground Truth Files (Exported from Transkribus)

In [11]:
gt_rows = parse_one_file(
    "/Users/sophiehamann/master-thesis-code/data/ground_truth/heresies_ground_truth/combined_xml/combined.xml",
    namespace
)

df_gt = pd.DataFrame(gt_rows)
print(df_gt.shape)
print(df_gt.head())

(1490, 3)
                                         source_file     region_type  \
0  /Users/sophiehamann/master-thesis-code/data/gr...     page_number   
1  /Users/sophiehamann/master-thesis-code/data/gr...         heading   
2  /Users/sophiehamann/master-thesis-code/data/gr...  author_creator   
3  /Users/sophiehamann/master-thesis-code/data/gr...            poem   
4  /Users/sophiehamann/master-thesis-code/data/gr...       paragraph   

                                                text  
0                                                 30  
1  The Art of Not Bowing: Writing by Women in Prison  
2                                        Carol Muske  
3                           Who the hell am I anyway  
4    In July 1973 I wrote an article for The Village  


#### 4. Loading the Parsed Corpus

In [12]:
df_ocr = pd.read_csv("/Users/sophiehamann/master-thesis-code/data/output_files/01_parsed_corpus.csv")
print(df_ocr.shape)
print(df_ocr.head())

(16265, 4)
                                         source_file  page_number  \
0  /Users/sophiehamann/master-thesis-code/data/ra...          NaN   
1  /Users/sophiehamann/master-thesis-code/data/ra...          NaN   
2  /Users/sophiehamann/master-thesis-code/data/ra...          NaN   
3  /Users/sophiehamann/master-thesis-code/data/ra...          NaN   
4  /Users/sophiehamann/master-thesis-code/data/ra...          NaN   

        region_type                                               text  
0  issue_front_page                                              THIRD  
1         paragraph  Editorial Collective: Lula Mae Blocton, Yvonne...  
2         paragraph  Heresies is an idea-oriented journal devoted t...  
3         paragraph  Heresies is structured as a collective of femi...  
4         paragraph  As women, we are aware that historically the c...  


#### 5. Calculating CER

In [13]:
from jiwer import cer, process_characters

gt_text  = " ".join(df_gt["text"].dropna().tolist())
ocr_text = " ".join(df_ocr["text"].dropna().tolist())

# overall CER
error_rate = cer(gt_text, ocr_text)
print(f"Character Error Rate: {error_rate:.2%}")

# breakdown by error type
details = process_characters(gt_text, ocr_text)
print(f"Insertions:     {details.insertions}")
print(f"Substitutions:  {details.substitutions}")
print(f"Deletions:      {details.deletions}")
print(f"Total characters in ground truth: {details.hits + details.deletions + details.substitutions}")

Character Error Rate: 918.61%
Insertions:     560132
Substitutions:  8449
Deletions:      10
Total characters in ground truth: 61897


Using this: 

https://blog.transkribus.org/en/how-is-the-cer-calculated-in-transkribus

and jiwer documentation: 

https://jitsi.github.io/jiwer/usage/

#### 6. Saving the Results

In [14]:
results = pd.DataFrame([{
    "total_gt_regions":  len(df_gt),
    "total_ocr_regions": len(df_ocr),
    "CER":               error_rate
}])

results.to_csv("/Users/sophiehamann/master-thesis-code/data/output_files/02_cer_results.csv", index=False)
print(results)

   total_gt_regions  total_ocr_regions       CER
0              1490              16265  9.186083


#### 7. Quick interpretation: 

https://blog.transkribus.org/en/how-is-the-cer-calculated-in-transkribus

I have a CER under 10% (see Official documentation in the link above.)


CHANGE THIS NUMBER AS SOON AS I HAVE EVERYTHING DIGITIZED